In [1]:
# ── Cell 1: Imports ──────────────────────────────────────────────────────────
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scienceplots                 # pip install SciencePlots
from datetime import datetime, timezone
plt.style.use(['science', 'no-latex'])

In [2]:

def unix_to_brisbane(unix_time):
    """Convert UNIX timestamp to Brisbane time (UTC+10)"""
    # Brisbane is UTC+10 (no daylight saving)
    brisbane_time = datetime.fromtimestamp(unix_time, tz=timezone.utc) + pd.Timedelta(hours=10)
    return brisbane_time.strftime('%Y-%m-%d %H:%M:%S')

In [80]:
# ── Cell 2: User Parameters ───────────────────────────────────────────────────
# ↓ Edit these three values before running the rest of the notebook

EVENTS_FOLDER   = r"C:\Arjun\Thesis\data\20200421_170039-sunset1\filtered chunks\un-normalised"
#EVENTS_FOLDER   = r"C:\Arjun\Thesis\data\20200421_170039-sunset1\filtered chunks"
START_TIMESTAMP = 1587452621.57     # start of integration window  (same units as CSV timestamps)
TIME_PERIOD     = 0.1               # duration of integration window (same units)

# Polarity filter  ← NEW: set to True (ON events) or False (OFF events) or None (both)
POLARITY_FILTER = True              # True = ON events only, False = OFF events only, None = all

# Derived stop threshold
END_TIMESTAMP = START_TIMESTAMP + TIME_PERIOD

# DAVIS resolution
WIDTH, HEIGHT = 346, 260

In [81]:
# ── Cell 3: Discover & sort CSV batch files ───────────────────────────────────
pattern   = os.path.join(EVENTS_FOLDER, "filtered_events_batch_*.csv")
#pattern   = os.path.join(EVENTS_FOLDER, "events_batch_*.csv")
csv_files = sorted(glob.glob(pattern))   # lexicographic sort matches zero-padded names

if not csv_files:
    raise FileNotFoundError(f"No CSV files matched: {pattern}")

print(f"Found {len(csv_files)} batch files.")
print("  First:", os.path.basename(csv_files[0]))
print("  Last :", os.path.basename(csv_files[-1]))

Found 162 batch files.
  First: filtered_events_batch_00000.csv
  Last : filtered_events_batch_00161.csv


In [82]:
# ── Cell 4: Load & filter events from CSV batches ────────────────────────────
# Collects (x, y, timestamp) for the chosen polarity within the time window.
# NOTE: This cell REPLACES the incomplete accumulation loop from Cell 3.

xs_all, ys_all, ts_all = [], [], []
stop_early = False

for csv_path in csv_files:
    if stop_early:
        break

    df = pd.read_csv(
        csv_path,
        usecols=['x', 'y', 'polarity', 'timestamp'],
        dtype={'timestamp': np.float64, 'x': np.int16, 'y': np.int16, 'polarity': str}
    )

    # Skip entire file if it ends before our window
    if df['timestamp'].max() < START_TIMESTAMP:
        continue

    # Stop after this file if it starts beyond our window
    if df['timestamp'].min() > END_TIMESTAMP:
        stop_early = True
        break

    # Trim to the time window
    mask = (df['timestamp'] >= START_TIMESTAMP) & (df['timestamp'] < END_TIMESTAMP)
    if (df['timestamp'] >= END_TIMESTAMP).any():
        stop_early = True
    df_win = df[mask].copy()

    if df_win.empty:
        continue

    # ── Polarity filter ───────────────────────────────────────────────────────
    pol_bool = df_win['polarity'].str.strip().str.upper() == 'TRUE'   # True = ON

    if POLARITY_FILTER is True:
        df_win = df_win[pol_bool]
    elif POLARITY_FILTER is False:
        df_win = df_win[~pol_bool]
    # else POLARITY_FILTER is None → keep all

    if df_win.empty:
        continue

    xs_all.append(df_win['x'].to_numpy().clip(0, WIDTH  - 1))
    ys_all.append(df_win['y'].to_numpy().clip(0, HEIGHT - 1))
    ts_all.append(df_win['timestamp'].to_numpy())

    print(f"  {os.path.basename(csv_path)} — {len(df_win)} events kept")

# Concatenate all batches
xs = np.concatenate(xs_all) if xs_all else np.array([], dtype=np.int16)
ys = np.concatenate(ys_all) if ys_all else np.array([], dtype=np.int16)
ts = np.concatenate(ts_all) if ts_all else np.array([], dtype=np.float64)

# Normalise timestamps to start at 0 (ms) for readable axis labels
#ts_ms = (ts - ts.min()) * 1000 if len(ts) else ts
ts_ms = (ts - START_TIMESTAMP) * 1000  

pol_label = {True: "ON (TRUE)", False: "OFF (FALSE)", None: "ALL"}[POLARITY_FILTER]
print(f"\nTotal events loaded: {len(xs):,}  |  Polarity: {pol_label}")

  filtered_events_batch_00017.csv — 28 events kept
  filtered_events_batch_00018.csv — 17 events kept

Total events loaded: 45  |  Polarity: ON (TRUE)


In [89]:
# ── Cell 5: 3D Event-Stream Scatter Plot ─────────────────────────────────────
FONT_SIZE = 22
AXIS_SIZE = 23
with plt.style.context(['science', 'no-latex']):

    fig = plt.figure(figsize=(14, 8))
    ax  = fig.add_subplot(111, projection='3d')

    # ── down-sample for speed if data is very large ───────────────────────────
    MAX_POINTS = 50_000
    if len(xs) > MAX_POINTS:
        idx = np.random.choice(len(xs), MAX_POINTS, replace=False)
        xs_plot, ys_plot, ts_plot = xs[idx], ys[idx], ts_ms[idx]
        sampled = True
    else:
        xs_plot, ys_plot, ts_plot = xs, ys, ts_ms
        sampled = False

    '''scatter = ax.scatter(
        ts_plot, xs_plot, ys_plot,
        c=ts_plot,
        cmap='inferno',
        s=15,
        alpha=0.7,
        linewidths=0
    )'''
    scatter = ax.scatter(
        ts_plot, xs_plot, ys_plot,
        color="#031332",     # ← single colour, no cmap
        s=45,
        alpha=0.7,
        linewidths=0
    )

    # ── colour-bar ────────────────────────────────────────────────────────────
    #cbar = fig.colorbar(scatter, ax=ax,  pad=0.1, shrink=0.55)
    #cbar.set_label('Time (ms)', fontsize=17)
    #cbar.ax.tick_params(labelsize=18)

    # ── tick font sizes ───────────────────────────────────────────────────────
    ax.tick_params(axis='x', labelsize=AXIS_SIZE )
    ax.tick_params(axis='y', labelsize=AXIS_SIZE ,labelpad=20)
    ax.tick_params(axis='z', labelsize=AXIS_SIZE )

    # ── axis labels ───────────────────────────────────────────────────────────
    ax.set_xlabel('Time (ms)', fontsize=AXIS_SIZE, labelpad=20)
    ax.set_ylabel('x (px)', fontsize=AXIS_SIZE, labelpad=20)
    #ax.set_ylabel('x (px)',    fontsize=17, labelpad=14)
    ax.set_zlabel('y (px)',    fontsize=AXIS_SIZE, labelpad=20)

    '''ax.set_xlim(0, TIME_PERIOD * 1000)
    ax.set_ylim(WIDTH  - 1, 0)     # 345 → 0  (flipped)
    ax.set_zlim(HEIGHT - 1, 0) 
    #ax.set_ylim(0, WIDTH  - 1)
    #ax.set_zlim(0, HEIGHT - 1)
    ax.invert_zaxis()'''

    ax.set_xlim(0, TIME_PERIOD * 1000)
    ax.set_xticks([0, 20, 40, 60, 80])
    ax.set_ylim(0, WIDTH  - 1)
    ax.set_zlim(0, HEIGHT - 1)
    ax.set_yticks([0, 100, 200, 300])
    ax.invert_zaxis()

    # ── event-count annotation ────────────────────────────────────────────────
    n_total = len(xs)


    # ── title ─────────────────────────────────────────────────────────────────
    pol_label = {True: 'ON (TRUE)', False: 'OFF (FALSE)', None: 'ALL'}[POLARITY_FILTER]
    fig.text(
        0.55, 0.97,
        f'Event stream with {pol_label} polarity',
        ha='center', va='top',
        fontsize=18
    )
    fig.text(
        0.55, 0.91,
        f'traverse: sunset1     window: 100 ms from {unix_to_brisbane(START_TIMESTAMP)}\n Events in window: {n_total:,}',
        ha='center', va='top',
        fontsize=FONT_SIZE - 4
    )

    ax.set_title('')

    # ── viewing angle ─────────────────────────────────────────────────────────
    ax.view_init(elev=20, azim=-40)

    plt.tight_layout(rect=[0, 0, 1, 0.90])
    plt.savefig('event_stream_3d.pdf', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved → event_stream_3d.pdf  ({n_total:,} events in window)")

ValueError: keyword labelpad is not recognized; valid keywords are ['size', 'width', 'color', 'tickdir', 'pad', 'labelsize', 'labelcolor', 'labelfontfamily', 'zorder', 'gridOn', 'tick1On', 'tick2On', 'label1On', 'label2On', 'length', 'direction', 'left', 'bottom', 'right', 'top', 'labelleft', 'labelbottom', 'labelright', 'labeltop', 'labelrotation', 'grid_agg_filter', 'grid_alpha', 'grid_animated', 'grid_antialiased', 'grid_clip_box', 'grid_clip_on', 'grid_clip_path', 'grid_color', 'grid_dash_capstyle', 'grid_dash_joinstyle', 'grid_dashes', 'grid_data', 'grid_drawstyle', 'grid_figure', 'grid_fillstyle', 'grid_gapcolor', 'grid_gid', 'grid_in_layout', 'grid_label', 'grid_linestyle', 'grid_linewidth', 'grid_marker', 'grid_markeredgecolor', 'grid_markeredgewidth', 'grid_markerfacecolor', 'grid_markerfacecoloralt', 'grid_markersize', 'grid_markevery', 'grid_mouseover', 'grid_path_effects', 'grid_picker', 'grid_pickradius', 'grid_rasterized', 'grid_sketch_params', 'grid_snap', 'grid_solid_ca

In [52]:
# ── Cell 6: 3D Event-Stream — Density Coloured ───────────────────────────────
from scipy.stats import gaussian_kde

FONT_SIZE = 22
AXIS_SIZE = 14

with plt.style.context(['science', 'no-latex']):

    fig = plt.figure(figsize=(12, 8))
    ax  = fig.add_subplot(111, projection='3d')

    # ── down-sample (same as Cell 5) ─────────────────────────────────────────
    MAX_POINTS = 50_000
    if len(xs) > MAX_POINTS:
        idx = np.random.choice(len(xs), MAX_POINTS, replace=False)
        xs_plot, ys_plot, ts_plot = xs[idx], ys[idx], ts_ms[idx]
    else:
        xs_plot, ys_plot, ts_plot = xs, ys, ts_ms

    # ── compute density ───────────────────────────────────────────────────────
    # KDE on a sub-sample for speed — normalise each axis to [0,1] first
    # so the bandwidth is not biased by axis scale differences
    ts_n = ts_plot / (TIME_PERIOD * 1000)          # normalise time  → [0, 1]
    xs_n = xs_plot / (WIDTH  - 1)                  # normalise x(px) → [0, 1]
    ys_n = ys_plot / (HEIGHT - 1)                  # normalise y(px) → [0, 1]

    # KDE sub-sample: gaussian_kde is O(n²) — 10k points is plenty for density
    KDE_MAX = 10_000
    if len(ts_n) > KDE_MAX:
        ki  = np.random.choice(len(ts_n), KDE_MAX, replace=False)
        kde = gaussian_kde(np.vstack([ts_n[ki], xs_n[ki], ys_n[ki]]))
    else:
        kde = gaussian_kde(np.vstack([ts_n, xs_n, ys_n]))

    # Evaluate density at every plotted point (batched to avoid memory spike)
    BATCH = 5_000
    density = np.empty(len(ts_n))
    for i in range(0, len(ts_n), BATCH):
        sl = slice(i, i + BATCH)
        density[sl] = kde(np.vstack([xs_n[sl], ys_n[sl]]))

    # ── scatter coloured by density ───────────────────────────────────────────
    # Sort by density ascending so dense points draw on top
    order   = np.argsort(density)
    scatter = ax.scatter(
        ts_plot[order], xs_plot[order], ys_plot[order],
        c=density[order],
        cmap='Blues',
        s=5,
        alpha=0.7,
        linewidths=0
    )

    # ── colour-bar ────────────────────────────────────────────────────────────
    cbar = fig.colorbar(scatter, ax=ax, pad=0.1, shrink=0.55)
    cbar.set_label('Density', fontsize=17)
    cbar.ax.tick_params(labelsize=18)

    # ── tick & axis labels ────────────────────────────────────────────────────
    ax.tick_params(axis='x', labelsize=AXIS_SIZE)
    ax.tick_params(axis='y', labelsize=AXIS_SIZE)
    ax.tick_params(axis='z', labelsize=AXIS_SIZE)

    ax.set_xlabel('Time (ms)', fontsize=17, labelpad=14)
    ax.set_ylabel('x (px)',    fontsize=17, labelpad=14)
    ax.set_zlabel('y (px)',    fontsize=17, labelpad=14)

    ax.set_xlim(0, TIME_PERIOD * 1000)
    ax.set_ylim(0, WIDTH  - 1)
    ax.set_zlim(0, HEIGHT - 1)
    ax.invert_zaxis()

    # ── titles ────────────────────────────────────────────────────────────────
    n_total   = len(xs)
    pol_label = {True: 'ON (TRUE)', False: 'OFF (FALSE)', None: 'ALL'}[POLARITY_FILTER]
    fig.text(
        0.55, 0.97,
        f'Event stream with {pol_label} polarity — density coloured',
        ha='center', va='top', fontsize=18
    )
    fig.text(
        0.55, 0.91,
        f'traverse: sunset1     window: 100 ms from {unix_to_brisbane(START_TIMESTAMP)}\n Events in window: {n_total:,}',
        ha='center', va='top', fontsize=FONT_SIZE - 4
    )
    ax.set_title('')

    ax.view_init(elev=20, azim=-40)

    plt.tight_layout(rect=[0, 0, 1, 0.90])
    plt.savefig('event_stream_3d_density.pdf', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved → event_stream_3d_density.pdf  ({n_total:,} events in window)")

ValueError: points have dimension 2, dataset has dimension 3